# AI-02 · Math and gradient descent

[Open in Colab](https://colab.research.google.com/github/BuiltBySwap/engineering-lab/blob/main/docs/topics/AI-02.ipynb)

**Status card** (edit it; the Progress and Tags pages are built from it):

```yaml
id: AI-02
title: "Math and gradient descent"
track: ai
date: 2026-10-09
status: in-progress  # not-started | in-progress | partial | done
score: ""
insight: ""
tags: [AI-02, ai, math, gradient-descent, python, numpy]
pending:
  - "Practice predictions for the three cosine pairs"
  - "Run notebook cells 2 to 4 (data, learning loop, three learning rates)"
  - "MQ433: how does an on-device model fit in phone memory (quantization)?"
  - "English: anomaly, culprit"
  - "Role lens: explain gradient descent to a founder in 3 sentences"
```

## Goal
Learning means adjusting numbers to reduce error. Write that loop myself for a straight line, and understand the two math tools behind it.

## What I learned (plain words)
- **Weights** are the adjustable numbers inside a model. **Loss** is one number for "how wrong". The **gradient** is the slope of the loss. The **learning rate** is the step size.
- Hiker in fog: feel the slope, step downhill, repeat.
- **Dot product:** multiply matching positions, then add. **Cosine similarity** divides by the two lengths, so only direction matters.

*Run each cell with **Shift + Enter**. Predict before you run.*

## 1 · Vectors, dot product, cosine similarity

A **vector** is a list of numbers. The **dot product** multiplies matching positions and adds them up. **Cosine similarity** divides that by the two lengths, so only the *direction* matters:

`cosine(u, v) = dot(u, v) / (length(u) * length(v))`

Worked example: `u = [1, 2, 3]`, `v = [3, -1, 0]`
- dot = 1×3 + 2×(−1) + 3×0 = **1**
- lengths: √14 ≈ 3.742 and √10 ≈ 3.162
- cosine = 1 ÷ (3.742 × 3.162) ≈ **0.085**

### The cosine line, piece by piece
```python
return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
```
| Piece | Meaning |
|---|---|
| `np.dot(u, v)` | dot product |
| `np.linalg.norm(u)` | length of the arrow `u` (Pythagoras) |
| `/` | divide; Python 3 always gives a decimal |

Worked example: `u = [1, 2, 3]`, `v = [3, -1, 0]`. Dot = 1×3 + 2×(−1) + 3×0 = 1. Lengths √14 ≈ 3.742 and √10 ≈ 3.162. Cosine = 1 ÷ 11.83 ≈ 0.085.

Cosine 1 means the same direction, 0 means unrelated, −1 means opposite. This is how semantic search compares meanings (AI-05).

In [ ]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([2, 4, 6])      # points the same way as a, just longer
c = np.array([3, -1, 0])     # points somewhere else

def cosine(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

print("a·b =", np.dot(a, b), " cosine =", round(cosine(a, b), 3))
print("a·c =", np.dot(a, c), " cosine =", round(cosine(a, c), 3))

### Where the formula comes from

**Wish list.** We want one number for "do these two arrows point the same way?" that
1. ignores how long the arrows are,
2. is 1 for the same direction, 0 for a right angle (unrelated) and −1 for opposite,
3. is cheap to compute (only multiplying and adding, no angles).

**Step 1: write each arrow with its angle.** An arrow of length `L` that points at angle `a` (measured from the x-axis) ends at the point `(L·cos a, L·sin a)`. So `u = (|u|·cos a, |u|·sin a)` and `v = (|v|·cos b, |v|·sin b)`.

**Step 2: take the dot product** (multiply matching positions, then add):

`u·v = |u||v|·cos a·cos b + |u||v|·sin a·sin b = |u||v|·(cos a·cos b + sin a·sin b)`

**Step 3: use a known identity from school trigonometry** (I use it, I do not prove it here):

`cos(a − b) = cos a·cos b + sin a·sin b`

So `u·v = |u||v|·cos(a − b)`. And `a − b` is exactly the angle θ between the two arrows. That gives the key fact:

**`u·v = |u| · |v| · cos θ`**

Divide both sides by `|u||v|` and you get the formula in the code:

**`cos θ = (u·v) / (|u| · |v|)`**

**Picture:** `|u|·cos θ` is the length of the *shadow* of `u` on `v`. The dot product is that shadow times `|v|`. Dividing by both lengths turns it into "what fraction of `u` lies along `v`", a pure direction number.

**Any number of dimensions:** any two arrows lie in one flat sheet (a 2-D plane), however many dimensions they live in. So the 2-D picture always holds, and the formula works for the 768-number vectors of an AI model too.

**Why cos θ and not θ itself?** Getting θ needs `arccos` (extra work), and cos θ already ranks pairs correctly: bigger cos θ means a smaller angle.

**Check it with numbers** (Python terms: `np.arctan2(y, x)` is the angle of the point `(x, y)` from the x-axis, like Kotlin's `Math.atan2(y, x)`; `np.degrees` converts radians to degrees, like `Math.toDegrees`).

In [ ]:
import numpy as np

u = np.array([3, 4])
v = np.array([4, 3])

angle_u = np.degrees(np.arctan2(u[1], u[0]))     # direction of u, measured from the x-axis
angle_v = np.degrees(np.arctan2(v[1], v[0]))     # direction of v
theta = angle_u - angle_v                          # the angle between the two arrows
print("angles:", round(angle_u, 2), round(angle_v, 2), "-> between them:", round(theta, 2), "degrees")

dot = np.dot(u, v)
length_u = np.linalg.norm(u)
length_v = np.linalg.norm(v)
print("dot product            :", dot)
print("|u| * |v| * cos(theta) :", round(length_u * length_v * np.cos(np.radians(theta)), 6))
print("cosine from the formula:", round(dot / (length_u * length_v), 6))
print("cos(theta) directly    :", round(np.cos(np.radians(theta)), 6))

### The same thing without NumPy (plain Python)

In [ ]:
import math

def dot(u, v):
    total = 0
    for i in range(len(u)):
        total += u[i] * v[i]
    return total

def length(u):
    return math.sqrt(dot(u, u))

def cosine_plain(u, v):
    return dot(u, v) / (length(u) * length(v))

print(cosine_plain([1, 2, 3], [3, -1, 0]))
assert abs(cosine_plain([1, 2, 3], [3, -1, 0]) - cosine(a, c)) < 1e-9   # same answer as NumPy

## Python traps
| Trap | Example |
|---|---|
| Lists do not multiply per element | `[1,2,3] * 2` repeats the list; a NumPy array gives `[2,4,6]` |
| `/` vs `//` | `7 / 2 = 3.5`, `7 // 2 = 3` |
| Rounding | `0.9999999999999998` instead of `1.0`; use `round(x, 3)`, never `==` on decimals |

### Practice: predict first

Write your three predictions here, then run the next cell.

- `[1, 0]` and `[0, 1]` → ?
- `[1, 1]` and `[2, 2]` → ?
- `[1, 0]` and `[-1, 0]` → ?

(Floating-point rounding may print `0.9999999999999998` instead of `1.0`. That is normal.)

In [ ]:
print(cosine(np.array([1, 0]), np.array([0, 1])))
print(cosine(np.array([1, 1]), np.array([2, 2])))
print(cosine(np.array([1, 0]), np.array([-1, 0])))

## 2 · Made-up data

The hidden rule is `y = 3x + 2`, plus noise. The model will not see the rule, only the points.

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
x = rng.uniform(0, 1, 50)
y = 3 * x + 2 + rng.normal(0, 0.3, 50)

plt.scatter(x, y)
plt.xlabel("x")
plt.ylabel("y")
plt.show()

## 3 · The learning loop

Each step:
1. predict with the current `w` and `b`
2. measure the **loss** (average squared error)
3. find the **slope** of the loss with respect to `w` and `b`
4. move each number a small step the other way (`learning rate` = step size)

In [ ]:
def fit(lr, steps):
    w, b = 0.0, 0.0                          # start knowing nothing
    losses = []
    for _ in range(steps):
        error = w * x + b - y                # how wrong is each prediction?
        losses.append((error ** 2).mean())   # loss = average squared error
        dw = 2 * (error * x).mean()          # slope of the loss with respect to w
        db = 2 * error.mean()                # slope of the loss with respect to b
        w -= lr * dw                         # step downhill
        b -= lr * db
    return w, b, losses

w, b, losses = fit(lr=0.1, steps=200)
print(f"w = {w:.2f}, b = {b:.2f}, final loss = {losses[-1]:.3f}")
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("loss")
plt.show()

## 4 · Three learning rates

**Predict first:** what will happen with 0.001, 0.1 and 1.0? Write one line each in the next cell before you run it.

In [ ]:
# My predictions:
# 0.001 -> ?
# 0.1   -> ?
# 1.0   -> ?

for lr in [0.001, 0.1, 1.0]:
    w, b, losses = fit(lr, steps=200)
    print(f"lr={lr:<6} w={w:.3g}  b={b:.3g}  loss: start={losses[0]:.2f}  end={losses[-1]:.3g}")

## Results (seed 42, 200 steps)
| Learning rate | What happened |
|---|---|
| 0.001 | Too slow: w ≈ 0.67, loss ≈ 4.9 |
| 0.1 | Good: w ≈ 2.90, b ≈ 2.01, loss ≈ 0.05 |
| 1.0 | Exploded: loss ≈ 1e83 |

✏️ Fill in after running the notebook. Even exact math gives about 3.04 and 1.93, not 3 and 2, because noise pulls the line.

## 5 · Check the slope formula (optional)

If the formula for the slope is right, nudging `w` a tiny bit should change the loss by about `slope × nudge`.

In [ ]:
def loss_at(w, b):
    return ((w * x + b - y) ** 2).mean()

w0, b0, eps = 1.0, 1.0, 1e-6
numeric = (loss_at(w0 + eps, b0) - loss_at(w0, b0)) / eps        # measured slope
analytic = 2 * ((w0 * x + b0 - y) * x).mean()                    # formula from the loop
print(numeric, analytic)

## Bugs and fixes
| Error or symptom | Cause | Fix |
|---|---|---|
| | | |

## Quiz and mistakes
- Score:

## In my own words
✏️ What is gradient descent, in three sentences a founder could follow?